# MQ-009: Medical RAG Retrieval Evaluation

## Research Objective

Evaluate semantic retrieval quality using the
MedQuAD medical knowledge corpus and FAISS.

## Previously Completed

- Cleaned medical records: 15,864
- Final medical passages: 38,544
- Embedding model: all-MiniLM-L6-v2
- Embedding dimensions: 384
- Vector index: FAISS IndexFlatIP
- Similarity: Cosine-equivalent inner product

## Notebook Scope

This notebook loads previously generated artifacts.

It does not repeat:
- Medical data preprocessing
- Passage chunking
- Embedding generation
- FAISS index construction

## Evaluation Plan

1. Load saved retrieval artifacts.
2. Load held-out validation questions.
3. Perform semantic retrieval.
4. Evaluate retrieval relevance.
5. Record experimental findings.

In [1]:
from pathlib import Path

import faiss
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(
    "/home/sagemaker-user/medical_chatbot"
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Load passage metadata
passages_df = pd.read_csv(
    PROCESSED_DIR / "medquad_final_passages.csv"
)

# Load existing FAISS index
faiss_index = faiss.read_index(
    str(PROCESSED_DIR / "medquad_faiss.index")
)

# Validate artifact consistency
assert len(passages_df) == faiss_index.ntotal
assert faiss_index.d == 384

print("Medical passages:", len(passages_df))
print("FAISS indexed vectors:", faiss_index.ntotal)
print("Embedding dimensions:", faiss_index.d)
print("Artifacts loaded successfully!")

Medical passages: 38544
FAISS indexed vectors: 38544
Embedding dimensions: 384
Artifacts loaded successfully!


In [2]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device="cpu"
)

print(
    "Embedding dimensions:",
    embedding_model.get_embedding_dimension()
)

print("Query embedding model ready!")

/home/sagemaker-user/medquad-envs/rag-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3012.33it/s]


Embedding dimensions: 384
Query embedding model ready!


In [3]:
def retrieve_medical_passages(query, top_k=5):

    # Encode only the new query
    query_vector = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        device="cpu"
    ).astype(np.float32)

    # Search existing FAISS index
    scores, indices = faiss_index.search(
        query_vector,
        top_k
    )

    results = []

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):
        if idx < 0:
            continue

        record = passages_df.iloc[int(idx)]

        results.append({
            "rank": rank,
            "similarity": float(score),
            "question": record["question"],
            "passage": record["passage"],
            "source": record["source"],
            "focus_area": record["focus_area"]
        })

    return pd.DataFrame(results)

In [4]:
query = "What are the symptoms of chronic kidney disease?"

results = retrieve_medical_passages(
    query,
    top_k=5
)

display(
    results[
        [
            "rank",
            "similarity",
            "question",
            "source",
            "focus_area"
        ]
    ]
)

,rank,similarity,question,source,focus_area
0,1,0.715603,What are the symptoms of Kidney Disease ?,NIHSeniorHealth,Kidney Disease
1,2,0.686266,What is (are) Chronic Kidney Disease ?,MPlusHealthTopics,Chronic Kidney Disease
2,3,0.677963,What are the symptoms of Atherosclerosis ?,NHLBI,Atherosclerosis
3,4,0.676642,What are the symptoms of Kidney Disease ?,NIHSeniorHealth,Kidney Disease
4,5,0.665614,What are the complications of Polycystic Kidne...,NIDDK,Polycystic Kidney Disease


## MQ-009: Retrieved Evidence Inspection

### Objective
Assess whether retrieved medical passages directly
answer the user's question.

### Evaluation Criteria
- Medical condition relevance
- Question-answer relevance
- Presence of useful clinical information
- Potentially misleading or unrelated evidence

### Research Importance
High embedding similarity does not necessarily imply
high clinical relevance.

The inspection will help identify whether additional
retrieval filtering or reranking is needed.

In [5]:
query = "What are the symptoms of chronic kidney disease?"

results = retrieve_medical_passages(query, top_k=5)

for row in results.itertuples(index=False):

    print(f"\nRank: {row.rank}")
    print(f"Similarity: {row.similarity:.4f}")
    print(f"Source: {row.source}")
    print(f"Focus area: {row.focus_area}")
    print(f"Original question: {row.question}")

    print("\nMedical evidence:")
    print(row.passage)

    print("\n" + "=" * 80)


Rank: 1
Similarity: 0.7156
Source: NIHSeniorHealth
Focus area: Kidney Disease
Original question: What are the symptoms of Kidney Disease ?

Medical evidence:
appetite, and some have leg cramps. These problems are caused by waste products that build up in the blood, a condition known as uremia. Healthy kidneys remove waste products from the blood. When the kidneys stop working, uremia occurs. The kidneys also make hormones and balance the minerals in the blood. When the kidneys stop working, most people develop conditions that affect the blood, bones, nerves, and skin. These problems may include itching, sleep problems, restless legs, weak bones, joint problems, and depression. How Kidney Disease Is Diagnosed Blood and urine tests are the only way to check for kidney damage or measure kidney function. It is important for you to get checked for kidney disease if you have the


Rank: 2
Similarity: 0.6863
Source: MPlusHealthTopics
Focus area: Chronic Kidney Disease
Original question: What

In [6]:
from pathlib import Path

# Retrieve baseline results
query = "What are the symptoms of chronic kidney disease?"

results = retrieve_medical_passages(
    query,
    top_k=5
)

# Create evaluation output directory
output_dir = Path(
    "/home/sagemaker-user/medical_chatbot/"
    "data/processed/retrieval_evaluation"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

# Save full passages without truncation
output_path = output_dir / "ckd_baseline_retrieval.csv"

results.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("Saved:", output_path)
print("Retrieved passages:", len(results))

# Display one passage at a time to avoid output truncation
for _, row in results.iterrows():
    print(
        f"Rank {row['rank']} | "
        f"Similarity {row['similarity']:.4f} | "
        f"{row['focus_area']}"
    )
    print("Passage length:", len(row["passage"]))

Saved: /home/sagemaker-user/medical_chatbot/data/processed/retrieval_evaluation/ckd_baseline_retrieval.csv
Retrieved passages: 5
Rank 1 | Similarity 0.7156 | Kidney Disease
Passage length: 727
Rank 2 | Similarity 0.6863 | Chronic Kidney Disease
Passage length: 689
Rank 3 | Similarity 0.6780 | Atherosclerosis
Passage length: 754
Rank 4 | Similarity 0.6766 | Kidney Disease
Passage length: 719
Rank 5 | Similarity 0.6656 | Polycystic Kidney Disease
Passage length: 677


## MQ-010: Held-Out Retrieval Evaluation

### Objective
Evaluate the baseline FAISS retriever using 50 held-out
medical questions.

### Dataset
- Validation dataset: validation.csv
- Storage: Amazon S3
- Retrieval corpus: 38,544 indexed passages

### Evaluation Strategy
The validation questions will be encoded using the
existing sentence-transformer model.

Retrieved passages will be assessed for medical topic
relevance and answer usefulness.

### Research Limitations
A matching medical focus area does not guarantee that
a passage correctly answers the question.

Human relevance judgments or independently verified
reference evidence will be needed for stronger
retrieval-quality conclusions.

In [7]:
import boto3
import pandas as pd
from io import BytesIO

s3 = boto3.client("s3")

response = s3.get_object(
    Bucket="medquad-ai-cleaned",
    Key="validation.csv"
)

validation_df = pd.read_csv(
    BytesIO(response["Body"].read())
)

print("Validation records:", len(validation_df))
print("Columns:", validation_df.columns.tolist())

print("\nMissing values:")
print(validation_df.isnull().sum())

print("\nSample validation questions:")
display(
    validation_df[
        ["question", "focus_area"]
    ].head(10)
)

Validation records: 50
Columns: ['question', 'answer', 'source', 'focus_area', 'input_text', 'target_text', 'input_token_length', 'target_token_length']

Missing values:
question               0
answer                 0
source                 0
focus_area             0
input_text             0
target_text            0
input_token_length     0
target_token_length    0
dtype: int64

Sample validation questions:


,question,focus_area
0,Is Galactosialidosis inherited ?,Galactosialidosis
1,What are the symptoms of Coronary Microvascula...,Coronary Microvascular Disease
2,What is (are) Schwartz-Jampel syndrome ?,Schwartz-Jampel syndrome
3,What is (are) Menetrier disease ?,Menetrier disease
4,What causes Hanhart syndrome ?,Hanhart syndrome
5,What is (are) Breast Cancer ?,Breast Cancer
6,What are the treatments for MTHFR gene mutation ?,MTHFR gene mutation
7,What are the treatments for aromatic l-amino a...,aromatic l-amino acid decarboxylase deficiency
8,What is (are) Sinusitis ?,Sinusitis
9,What are the treatments for Crimean-Congo Hemo...,Crimean-Congo Hemorrhagic Fever (CCHF)


## MQ-010: Baseline Retrieval Topic-Match Evaluation

### Objective
Measure whether FAISS retrieves passages associated
with the same medical focus area as held-out questions.

### Evaluation Dataset
50 validation questions from MedQuAD.

### Metrics
- Focus Area Match@1
- Focus Area Match@3
- Focus Area Match@5

A match occurs when at least one retrieved passage
has the same normalized focus-area label as the query.

### Limitations
Focus-area matching is an imperfect relevance proxy.
Related conditions may have different labels, while
same-topic passages may not answer the question.

These metrics must not be interpreted as clinical
factuality, answer correctness, or true Recall@K.

In [8]:
import numpy as np
import pandas as pd

# Normalize focus-area labels
def normalize_focus_area(value):
    return " ".join(str(value).casefold().split())


# Encode only the 50 validation questions
validation_vectors = embedding_model.encode(
    validation_df["question"].tolist(),
    batch_size=32,
    convert_to_numpy=True,
    normalize_embeddings=True,
    device="cpu",
    show_progress_bar=True
).astype(np.float32)

# Search existing FAISS index
scores, indices = faiss_index.search(
    validation_vectors,
    k=5
)

evaluation_records = []

for i, row in validation_df.iterrows():

    expected_focus = normalize_focus_area(
        row["focus_area"]
    )

    retrieved_focus_areas = [
        normalize_focus_area(
            passages_df.iloc[int(idx)]["focus_area"]
        )
        for idx in indices[i]
        if idx >= 0
    ]

    matches = [
        focus == expected_focus
        for focus in retrieved_focus_areas
    ]

    evaluation_records.append({
        "question": row["question"],
        "expected_focus_area": row["focus_area"],
        "match_at_1": any(matches[:1]),
        "match_at_3": any(matches[:3]),
        "match_at_5": any(matches[:5]),
        "top_1_similarity": float(scores[i][0]),
        "retrieved_focus_areas": " | ".join(
            retrieved_focus_areas
        )
    })

evaluation_df = pd.DataFrame(evaluation_records)

print("Evaluated questions:", len(evaluation_df))

print(
    "Focus Area Match@1:",
    f"{evaluation_df['match_at_1'].mean():.2%}"
)

print(
    "Focus Area Match@3:",
    f"{evaluation_df['match_at_3'].mean():.2%}"
)

print(
    "Focus Area Match@5:",
    f"{evaluation_df['match_at_5'].mean():.2%}"
)

print("\nQuestions without a matching focus area in Top 5:")

display(
    evaluation_df.loc[
        ~evaluation_df["match_at_5"],
        [
            "question",
            "expected_focus_area",
            "retrieved_focus_areas"
        ]
    ].head(10)
)

Batches: 100%|██████████| 2/2 [00:00<00:00,  4.34it/s]


Evaluated questions: 50
Focus Area Match@1: 66.00%
Focus Area Match@3: 72.00%
Focus Area Match@5: 82.00%

Questions without a matching focus area in Top 5:


,question,expected_focus_area,retrieved_focus_areas
5,What is (are) Breast Cancer ?,Breast Cancer,male breast cancer | male breast cancer | male...
8,What is (are) Sinusitis ?,Sinusitis,preauricular sinus | preauricular sinus | sick...
11,What are the genetic changes related to mucopo...,mucopolysaccharidosis type II,mucopolysaccharidosis type iv | mucopolysaccha...
12,What is (are) Dendritic cell tumor ?,Dendritic cell tumor,desmoplastic small round cell tumor | paraneop...
13,What is (are) Pinworms ?,Pinworms,parasites - enterobiasis (also known as pinwor...
21,Do you have information about Child Care,Child Care,premature babies | children's health | prenata...
42,What are the symptoms of Madelung disease ?,Madelung disease,whipple disease | churg strauss syndrome | bal...
43,Do you have information about Prostate Cancer ...,Prostate Cancer Screening,prostate cancer | prostate cancer | prostate c...
47,What are the symptoms of Cobb syndrome ?,Cobb syndrome,beare-stevenson cutis gyrata syndrome | townes...


## MQ-011: Retrieval Failure Diagnosis

### Objective
Determine whether unmatched validation questions
represent missing knowledge coverage or retrieval
ranking problems.

### Methodology
1. Identify validation questions without a matching
   focus area in the top five results.
2. Search the complete retrieval metadata for an
   exact normalized focus-area match.
3. Count available passages for each medical topic.

### Interpretation
- Zero matching passages: Potential coverage or
  terminology mismatch.
- One or more matching passages: The topic is
  represented, but FAISS did not retrieve its exact
  focus-area label in the top five.

### Limitation
Exact focus-area matching cannot identify medical
synonyms or verify passage-level clinical relevance.

In [9]:
# MQ-011: Diagnose unmatched validation questions

corpus_focus = (
    passages_df["focus_area"]
    .fillna("")
    .map(normalize_focus_area)
)

unmatched_df = evaluation_df.loc[
    ~evaluation_df["match_at_5"]
].copy()

diagnostic_records = []

for row in unmatched_df.itertuples(index=False):

    expected = normalize_focus_area(
        row.expected_focus_area
    )

    available_passages = int(
        (corpus_focus == expected).sum()
    )

    diagnostic_records.append({
        "question": row.question,
        "expected_focus_area": row.expected_focus_area,
        "available_passages": available_passages,
        "exact_focus_area_in_corpus": (
            available_passages > 0
        ),
        "diagnosis": (
            "Potential ranking failure"
            if available_passages > 0
            else "Coverage or terminology mismatch"
        )
    })

diagnostic_df = pd.DataFrame(diagnostic_records)

display(diagnostic_df)

print("\nTotal unmatched:", len(diagnostic_df))

print(
    "Topics with exact focus-area coverage:",
    diagnostic_df["exact_focus_area_in_corpus"].sum()
)

print(
    "Topics without exact focus-area coverage:",
    (~diagnostic_df["exact_focus_area_in_corpus"]).sum()
)

,question,expected_focus_area,available_passages,exact_focus_area_in_corpus,diagnosis
0,What is (are) Breast Cancer ?,Breast Cancer,273,True,Potential ranking failure
1,What is (are) Sinusitis ?,Sinusitis,0,False,Coverage or terminology mismatch
2,What are the genetic changes related to mucopo...,mucopolysaccharidosis type II,8,True,Potential ranking failure
3,What is (are) Dendritic cell tumor ?,Dendritic cell tumor,0,False,Coverage or terminology mismatch
4,What is (are) Pinworms ?,Pinworms,0,False,Coverage or terminology mismatch
5,Do you have information about Child Care,Child Care,0,False,Coverage or terminology mismatch
6,What are the symptoms of Madelung disease ?,Madelung disease,8,True,Potential ranking failure
7,Do you have information about Prostate Cancer ...,Prostate Cancer Screening,0,False,Coverage or terminology mismatch
8,What are the symptoms of Cobb syndrome ?,Cobb syndrome,0,False,Coverage or terminology mismatch



Total unmatched: 9
Topics with exact focus-area coverage: 3
Topics without exact focus-area coverage: 6


## MQ-012: Retrieval Ranking Depth Analysis

### Objective
Determine whether medical passages associated with
the correct focus area appear beyond the top five
retrieval results.

### Methodology
1. Select questions with exact focus-area coverage.
2. Retrieve the top 50 passages using the existing
   FAISS index.
3. Identify the rank of the first matching focus area.
4. Record the similarity score at that rank.

### Research Significance
If relevant topic labels appear at deeper ranks,
reranking a larger candidate pool may improve
top-five retrieval performance.

### Limitations
Matching a focus area does not guarantee that a
passage answers the question or is clinically correct.

In [10]:
# MQ-012: Analyze retrieval depth

ranking_candidates = diagnostic_df[
    diagnostic_df["exact_focus_area_in_corpus"]
].copy()

ranking_results = []

for row in ranking_candidates.itertuples(index=False):

    query_vector = embedding_model.encode(
        [row.question],
        convert_to_numpy=True,
        normalize_embeddings=True,
        device="cpu"
    ).astype(np.float32)

    scores, indices = faiss_index.search(
        query_vector,
        k=50
    )

    expected_focus = normalize_focus_area(
        row.expected_focus_area
    )

    first_match_rank = None
    first_match_score = None

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):
        if idx < 0:
            continue

        retrieved_focus = normalize_focus_area(
            passages_df.iloc[int(idx)]["focus_area"]
        )

        if retrieved_focus == expected_focus:
            first_match_rank = rank
            first_match_score = float(score)
            break

    ranking_results.append({
        "question": row.question,
        "focus_area": row.expected_focus_area,
        "available_passages": row.available_passages,
        "first_match_rank": first_match_rank,
        "first_match_similarity": first_match_score
    })

ranking_depth_df = pd.DataFrame(ranking_results)

display(ranking_depth_df)

,question,focus_area,available_passages,first_match_rank,first_match_similarity
0,What is (are) Breast Cancer ?,Breast Cancer,273,8,0.644317
1,What are the genetic changes related to mucopo...,mucopolysaccharidosis type II,8,9,0.707559
2,What are the symptoms of Madelung disease ?,Madelung disease,8,6,0.573008


## MQ-013: Cross-Encoder Reranking Experiment

### Research Question
Can cross-encoder reranking improve the relevance
of retrieved medical evidence compared with
FAISS cosine-similarity ranking alone?

### Baseline
- Embedding model: all-MiniLM-L6-v2
- Vector index: FAISS IndexFlatIP
- Focus Area Match@5: 82%

### Experimental Method
1. Retrieve 20 candidate passages using FAISS.
2. Score each query-passage pair with a cross-encoder.
3. Rerank candidates by cross-encoder score.
4. Select the top five passages.
5. Compare results against the existing FAISS baseline.

### Model
cross-encoder/ms-marco-MiniLM-L-6-v2

### Limitations
The reranker is trained primarily for general
information retrieval, not clinical reasoning.

Higher topic-match scores do not necessarily imply
greater medical factuality or safety.

In [11]:
from sentence_transformers import CrossEncoder

RERANKER_MODEL_NAME = (
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

reranker = CrossEncoder(
    RERANKER_MODEL_NAME,
    device="cpu"
)

print("Reranker loaded successfully!")
print("Model:", RERANKER_MODEL_NAME)
print("Device: CPU")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 19652.03it/s]


Reranker loaded successfully!
Model: cross-encoder/ms-marco-MiniLM-L-6-v2
Device: CPU
